# Day 49：里程碑项目 3（Part 2）—— 多 Agent 智能办公助手全栈实战

> **🎙️ 讲师旁白**：欢迎来到星辰数科《智枢企业级智能体平台》第四阶段（Day 49）！今天我们将在 Notebook 中通过 FastAPI `TestClient`（进程内调用，无需 `input()`）全链路演练 **多 Agent 智能办公助手（`v0.4.0-day49`）**：包括前端页面托管、低风险只读任务直通、高风险任务 HITL 中断挂起（`awaiting_approval`）、人工批准微调恢复（`resume`）以及人工拒绝安全拦截。

## 1. 环境初始化与启动隔离 FastAPI `TestClient`
定位仓库根目录并将 `project_iterations/day_49` 加入 `sys.path`，使用临时目录构建隔离的 SQLite 检查点数据库。

In [1]:
import json
import sys
import tempfile
from pathlib import Path

from fastapi.testclient import TestClient

repo_root = Path.cwd()
while not (repo_root / "project_iterations").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
day49_dir = repo_root / "project_iterations" / "day_49"
if str(day49_dir) not in sys.path:
    sys.path.insert(0, str(day49_dir))

from zhishu_agent.office_web_app import create_app

_tmp_dir = tempfile.TemporaryDirectory()
db_path = Path(_tmp_dir.name) / "demo_office.db"
app = create_app(db_path=db_path, frontend_dir=day49_dir / "frontend")
client = TestClient(app)

health = client.get("/api/v1/health").json()
home = client.get("/")
print("已加载模块路径:", day49_dir)
print("健康检查:", health, "| 首页状态码:", home.status_code)

已加载模块路径: /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_49
健康检查: {'status': 'ok', 'version': 'v0.4.0-day49'} | 首页状态码: 200


/home/antigravity/ab6539763/-AI-Agent/.venv/lib/python3.11/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## 2. 低风险只读检索任务（免审批直通 `completed`）
演示 Supervisor 将制度查询请求路由至 `kb_researcher`，无需触发人工中断直接完成。

In [2]:
r_low = client.post(
    "/api/v1/office/run",
    json={"task": "查询公司 2026 差旅报销与年假制度", "user_id": "chenmo"},
)
low_body = r_low.json()
print("状态:", low_body["status"], "| Thread ID:", low_body["thread_id"])
for step in low_body["handoff_trace"]:
    print(f"  #{step['step']} {step['from_agent']} -> {step['to_agent']} [{step['action']}]")
print("最终输出:\n" + low_body["final_output"])

状态: completed | Thread ID: thread-9b7713aeb1d3
  #1 user -> supervisor [receive_task]
  #2 supervisor -> kb_researcher [delegate_research]
  #3 kb_researcher -> supervisor [return_findings]
  #4 supervisor -> user [complete_task]
最终输出:
【智能办公调研报告】
针对您的请求「查询公司 2026 差旅报销与年假制度」，`kb_researcher` 已检索企业制度库并完成整理：
1. 核心依据：星辰数科《2026 智能办公与差旅规范 v3.2》；
2. 执行建议：本任务属于只读信息检索与摘要生成，不涉及外部发送或资金审批，已自动完成。


## 3. 高风险任务 HITL 中断挂起与人工批准恢复（`awaiting_approval` → `completed`）
发送全员邮件属于不可逆外部副作用操作，系统将在 `email_dispatcher` 执行前挂起线程；随后通过 `POST /api/v1/office/resume` 批准并微调主题完成闭环。

In [3]:
r_high = client.post(
    "/api/v1/office/run",
    json={"task": "发送全员邮件通知本周五晚 20:00 进行系统升级维护", "user_id": "linxiao"},
)
high_body = r_high.json()
tid = high_body["thread_id"]
print("挂起状态:", high_body["status"], "| Thread ID:", tid)
print("待审批动作:", json.dumps(high_body["pending_action"], ensure_ascii=False, indent=2))

r_resume = client.post(
    "/api/v1/office/resume",
    json={
        "thread_id": tid,
        "approved": True,
        "comment": "已核对维护窗口，准予发送",
        "edited_payload": {"subject": "【运维公告】本周五晚 20:00 系统升级通知"},
    },
)
resumed = r_resume.json()
print("恢复后状态:", resumed["status"], "| 决策:", resumed["decision"])
print("最终输出:\n" + resumed["final_output"])

挂起状态: awaiting_approval | Thread ID: thread-b78c687a2b17
待审批动作: {
  "action_id": "act-c6702cc957",
  "agent": "email_dispatcher",
  "action_type": "send_email",
  "risk_level": "high",
  "title": "高风险动作审批：对外/全员发送邮件",
  "description": "准备执行邮件发送任务：「发送全员邮件通知本周五晚 20:00 进行系统升级维护」。邮件发出后不可撤回，请人工确认收件人与正文内容。",
  "payload": {
    "to": [
      "all-staff@xingchen.com"
    ],
    "subject": "【办公助手代发】发送全员邮件通知本周五晚 20:00 进行系统升级维护",
    "body": "各位同事好，\n\n根据指令「发送全员邮件通知本周五晚 20:00 进行系统升级维护」整理通知如下，请查收。\n\n—— 智枢智能办公助手"
  }
}
恢复后状态: completed | 决策: approved
最终输出:
✅ 【人工审批通过 · 任务已执行】
动作类型：send_email（由 email_dispatcher 执行）
执行参数：{"to": ["all-staff@xingchen.com"], "subject": "【运维公告】本周五晚 20:00 系统升级通知", "body": "各位同事好，\n\n根据指令「发送全员邮件通知本周五晚 20:00 进行系统升级维护」整理通知如下，请查收。\n\n—— 智枢智能办公助手"}
审批备注：已核对维护窗口，准予发送
执行结果：操作已成功落盘并完成闭环通知。


## 4. 人工拒绝安全拦截与线程状态持久化回查
演示对高风险财务审批任务执行 `approved=False` 拦截，并通过 `GET /api/v1/office/threads/{thread_id}` 回查持久化状态。

In [4]:
r_exp = client.post(
    "/api/v1/office/run",
    json={"task": "审批张工的差旅报销申请 2480 元"},
).json()
exp_tid = r_exp["thread_id"]

r_reject = client.post(
    "/api/v1/office/resume",
    json={"thread_id": exp_tid, "approved": False, "comment": "缺少发票附件，暂不批准"},
).json()

persisted = client.get(f"/api/v1/office/threads/{exp_tid}").json()
print("拦截后持久化状态:", persisted["status"], "| 决策:", persisted["decision"])
print(persisted["final_output"])
_tmp_dir.cleanup()

拦截后持久化状态: rejected | 决策: rejected
⛔ 【人工审批已拒绝 · 操作已安全拦截】
被拦截动作：高风险动作审批：财务报销/流程签批（approve_expense）
拒绝原因：缺少发票附件，暂不批准
系统状态：未产生任何外部副作用，线程已安全归档。
